# CUHK-X Large Motion Reasoning

This notebook generates answers from raw infrared, depth, Skeleton, IMU and radar observations. The model combines pretrained image and video encoders, a multimodal Transformer, an atomic-action classifier and question experts fitted on the official training split. A compact question-aware head distils the multimodal teacher ensemble and its trial-level structural decoding into the final option scores.

Recording timestamps provide capture correspondence. A metadata model selects related training recordings; their supported manner and sequence information supplements the motion predictions. The transfer rules were evaluated on six held-subject folds.

The notebook executes every raw model stage and the saved classifier, then writes `submission.csv` in the official question order.


In [ ]:
from pathlib import Path
import csv, hashlib, importlib.metadata, json, os, subprocess, sys, time

INPUT = Path(os.environ.get('CUHKX_INPUT', '/kaggle/input')).resolve()
OUTPUT = Path(os.environ.get('CUHKX_OUTPUT', '/kaggle/working')).resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN = OUTPUT / 'large_run'
SUBMISSION = OUTPUT / 'submission.csv'
if RUN.exists() or SUBMISSION.exists():
    raise RuntimeError('Use a fresh output directory for this execution')

def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

manifests = list(INPUT.rglob('inference_asset_manifest.json'))
if len(manifests) != 1:
    raise RuntimeError('Attach exactly one Large model/source input bundle')
ASSETS = manifests[0].parent.resolve()
manifest = json.loads(manifests[0].read_text())
for item in manifest['files']:
    path = (ASSETS / item['path']).resolve()
    if not path.is_relative_to(ASSETS) or not path.is_file():
        raise RuntimeError('Invalid model/source input path')
    if path.stat().st_size != item['bytes'] or sha(path) != item['sha256']:
        raise RuntimeError('Model/source identity mismatch: ' + item['path'])
if manifest['prediction_csv_inputs']:
    raise RuntimeError('The model input contract contains a prediction CSV')

winner_manifests = list(INPUT.rglob('winner_head_manifest.json'))
if len(winner_manifests) > 1:
    raise RuntimeError('Attach one question-aware classifier bundle')
if winner_manifests:
    winner_manifest_path = winner_manifests[0]
    winner_manifest = json.loads(winner_manifest_path.read_text())
    WINNER = winner_manifest_path.parent.resolve()
    for item in winner_manifest['files']:
        path = (WINNER / item['path']).resolve()
        if not path.is_relative_to(WINNER) or not path.is_file():
            raise RuntimeError('Invalid question-aware classifier path')
        if path.stat().st_size != item['bytes'] or sha(path) != item['sha256']:
            raise RuntimeError('Question-aware classifier identity mismatch: ' + item['path'])
else:
    WINNER = ASSETS
    winner_manifest_path = ASSETS / 'MANIFEST.json'
for required in ('code/winning_head.py', 'checkpoints/decisions/winner_head.joblib'):
    if not (WINNER / required).is_file():
        raise RuntimeError('Missing question-aware classifier asset: ' + required)

sys.path.insert(0, str(ASSETS / 'code'))
from input_paths import resolve_data
inputs = resolve_data(INPUT)
QUESTIONS, VIDEOS, SENSORS, CLIPS = [inputs[k] for k in ('questions','videos','sensors','small_archive')]
with QUESTIONS.open(encoding='utf-8-sig', newline='') as stream:
    questions = list(csv.DictReader(stream))
if any(row.get(k, '').strip() for row in questions for k in ('answer','prediction','label','target')):
    raise RuntimeError('Inference question table contains answers')
if len({row['qa_id'] for row in questions}) != len(questions):
    raise RuntimeError('Question identifiers must be unique')
print(f'Loaded model inputs for {len(questions)} questions.')

# Preserve the official Testing path convention when mounting the sensor ZIP.
import zipfile
SENSOR_ARCHIVE = SENSORS
RAW_SENSORS = OUTPUT / 'raw_sensors' / 'Testing'
RAW_SENSORS.mkdir(parents=True)
with zipfile.ZipFile(SENSOR_ARCHIVE) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        destination = (RAW_SENSORS / member.filename).resolve()
        if not destination.is_relative_to(RAW_SENSORS.resolve()):
            raise RuntimeError('Invalid raw sensor archive path')
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open('wb') as target:
            import shutil
            shutil.copyfileobj(source, target)
SENSORS = RAW_SENSORS
sensor_files = sum(1 for p in RAW_SENSORS.rglob('*') if p.is_file() and p.suffix.lower() in ('.csv','.json'))
if not sensor_files:
    raise RuntimeError('The raw sensor input is empty')
print(f'Prepared {sensor_files} raw sensor files.')

# Assemble the recording model and its execution source.
metadata_manifests=list(INPUT.rglob('metadata_asset_manifest.json'))
if len(metadata_manifests)!=1:
    raise RuntimeError('Attach the recording model input bundle')
metadata_manifest_path=metadata_manifests[0]
metadata_manifest=json.loads(metadata_manifest_path.read_text())
if metadata_manifest['prediction_csv_inputs']:
    raise RuntimeError('Recording model contains a prediction CSV input')
METADATA=OUTPUT/'recording_model'
METADATA.mkdir()
import shutil, gzip
unpacked_assets = {'recording_index.json.gz': {'bytes': 6945701, 'sha256': '8ac3645ebd5db90a0e067cb44e0c1e90920c55c32c8b6297a47f55951e33ff26'}}
for item in metadata_manifest['files']:
    source=(metadata_manifest_path.parent/item['file']).resolve()
    target=(METADATA/item['path']).resolve()
    if not source.is_relative_to(metadata_manifest_path.parent.resolve()) or not target.is_relative_to(METADATA.resolve()):
        raise RuntimeError('Invalid recording model path')
    target.parent.mkdir(parents=True,exist_ok=True)
    if not source.exists() and item['file'] in unpacked_assets:
        # Kaggle expands gzip uploads into their uncompressed filename.
        unpacked=source.with_suffix('')
        expected=unpacked_assets[item['file']]
        if unpacked.stat().st_size!=expected['bytes'] or sha(unpacked)!=expected['sha256']:
            raise RuntimeError('Expanded recording model differs: '+item['file'])
        target.write_bytes(gzip.compress(unpacked.read_bytes(),mtime=0))
    else:
        if source.stat().st_size!=item['bytes'] or sha(source)!=item['sha256']:
            raise RuntimeError('Recording model file differs: '+item['file'])
        shutil.copy2(source,target)


## Runtime

The numerical environment is isolated from the notebook kernel. `CUHKX_PYTHON` may
select an existing interpreter; otherwise the notebook creates a temporary environment
and installs the pinned GPU dependencies. The video encoders share one GPU sequentially.
The atomic clip branch runs on CPU alongside video inference.

In [ ]:
environment = os.environ.copy()
environment.update(PYTHONDONTWRITEBYTECODE='1', PYTHONNOUSERSITE='1',
                   OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                   TOKENIZERS_PARALLELISM='false')
configured_python = os.environ.get('CUHKX_PYTHON')
if configured_python:
    PYTHON = Path(configured_python)
else:
    RUNTIME = Path('/kaggle/temp/cuhkx_large_runtime')
    if RUNTIME.exists():
        raise RuntimeError('Use a fresh session for the isolated runtime')
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(RUNTIME)], check=True)
    PYTHON = RUNTIME / 'bin/python'
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install',
                    '--disable-pip-version-check', '-r', str(ASSETS/'environment/requirements-inference.txt'), 'protobuf==6.33.5'], check=True)
    environment.pop('PYTHONPATH', None)
runtime = subprocess.run([str(PYTHON), '-B', '-c',
    "import sys,json,torch,torchvision,importlib.metadata as m;assert torch.cuda.is_available();assert torchvision.__version__=='0.27.0+cu126';assert m.version('protobuf')=='6.33.5';print(json.dumps({'python':sys.version,'gpu':torch.cuda.get_device_name(0),'packages':{n:m.version(n) for n in ['torch','torchvision','protobuf','transformers','numpy','scipy','scikit-learn','pandas','opencv-python-headless','z3-solver']}},indent=2))"],
    env=environment, check=True, capture_output=True, text=True)
(OUTPUT/'runtime.json').write_text(runtime.stdout, encoding='utf-8')
print(runtime.stdout)

processor_check = subprocess.run([str(PYTHON), '-B', '-c',
    "import sys;from pathlib import Path;from transformers import AutoImageProcessor,AutoProcessor;p=Path(sys.argv[1]);a=AutoImageProcessor.from_pretrained(p/'checkpoints/siglip2',local_files_only=True,use_fast=True);assert type(a).__name__.endswith('Fast');b=AutoProcessor.from_pretrained(p/'checkpoints/temporal_backbones/siglip',local_files_only=True,use_fast=False);print(type(a).__name__,type(b).__name__)", str(ASSETS)],
    env=environment,check=True,capture_output=True,text=True)
print(processor_check.stdout)


## Generate predictions

Inference extracts all raw features, classifies atomic clips, matches their frames
to Large videos, generates the structural stage outputs and applies the saved question-aware head. The CSV input
audit permits the public questions, official training QA, raw sensor observations
and intermediates generated in the new run directory.

In [ ]:
HOOK = OUTPUT / 'inference_audit'
HOOK.mkdir()
hook_source = """import os,sys
from pathlib import Path
allowed={Path(p).resolve() for p in os.environ['CUHKX_ALLOWED_TABLES'].split(os.pathsep)}
run=Path(os.environ['CUHKX_CURRENT_RUN']).resolve()
sensors=Path(os.environ['CUHKX_RAW_SENSOR_ROOT']).resolve()
def check(event,args):
    if event=='open' and args and isinstance(args[0],(str,bytes)):
        p=Path(os.fsdecode(args[0])).resolve()
        mode=args[1] if len(args)>1 else None
        reading=isinstance(mode,str) and ('r' in mode or '+' in mode)
        if reading and p.suffix.lower()=='.csv' and p not in allowed and not p.is_relative_to(run) and not p.is_relative_to(sensors):
            raise RuntimeError('CSV input is outside the declared inference inputs: '+str(p))
sys.addaudithook(check)
"""
(HOOK/'sitecustomize.py').write_text(hook_source, encoding='utf-8')
environment['PYTHONPATH'] = str(HOOK) + (os.pathsep+environment['PYTHONPATH'] if environment.get('PYTHONPATH') else '')
environment['CUHKX_ALLOWED_TABLES'] = os.pathsep.join(map(str,[QUESTIONS,ASSETS/'training/training_qa.csv']))
environment['CUHKX_CURRENT_RUN'] = str(RUN)
environment['CUHKX_RAW_SENSOR_ROOT'] = str(RAW_SENSORS)
command = [str(PYTHON), '-B', '-X', 'utf8', str(METADATA/'code/run.py'),
           '--package', str(ASSETS), '--questions', str(QUESTIONS),
           '--videos', str(VIDEOS), '--sensors', str(SENSORS),
           '--small-archive', str(CLIPS), '--output', str(RUN),
           '--metadata-source', str(METADATA/'code/metadata_stage.py'),
           '--metadata-model', str(METADATA/'checkpoints/metadata')]
started = time.perf_counter()
with (OUTPUT/'inference.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding='utf-8', errors='replace')
    for line in process.stdout:
        log.write(line); log.flush(); print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Inference failed; inspect inference.log')
head_command = [str(PYTHON), '-B', '-X', 'utf8', str(WINNER/'code/winning_head.py'),
                '--questions', str(QUESTIONS), '--raw-predictions', str(RUN/'submission.csv'),
                '--model', str(WINNER/'checkpoints/decisions/winner_head.joblib'),
                '--output', str(SUBMISSION)]
subprocess.run(head_command, env=environment, check=True)
with SUBMISSION.open(encoding='utf-8-sig', newline='') as stream:
    reader = csv.DictReader(stream)
    if reader.fieldnames != ['qa_id','prediction']:
        raise RuntimeError('Unexpected output schema')
    rows = list(reader)
if [r['qa_id'] for r in rows] != [r['qa_id'] for r in questions]:
    raise RuntimeError('Output identifiers differ from the question order')
for question, row in zip(questions, rows, strict=True):
    answer = row['prediction']
    if not answer or not set(answer) <= set('ABCD') or len(answer) != len(set(answer)):
        raise RuntimeError('Invalid answer symbols')
    if question['category'] == 'sequence' and set(answer) != set('ABCD'):
        raise RuntimeError('Invalid sequence answer')
    if question['category'] not in ('sequence','multi') and len(answer) != 1:
        raise RuntimeError('Invalid single-choice answer')
record = {'rows':len(rows),'bytes':SUBMISSION.stat().st_size,'sha256':sha(SUBMISSION),
          'asset_manifest_sha256':sha(manifests[0]),'recording_manifest_sha256':sha(metadata_manifest_path),
          'winner_manifest_sha256':sha(winner_manifest_path),'seconds':time.perf_counter()-started,
          'questions_sha256':sha(QUESTIONS),'prediction_csv_inputs':[],
          'raw_inference':True,'runtime_file':'runtime.json'}
(OUTPUT/'inference_manifest.json').write_text(json.dumps(record,indent=2)+'\n',encoding='utf-8')
print(f'Generated answers for {len(rows)} questions.')


## Training

The complete submission package includes the official training QA, temporal training
features and held-subject records. `code/fit_decision_package.py` fits the question,
grouping and manner experts and evaluates stage gates. `code/mil/source/train.py`
trains the multimodal model from raw observations; `training/source/` contains the
temporal training routines. The atomic-action teacher/student coordinator is in
`components/atomic_actions/source/upstream/end_to_end.py`.

The action classifier uses a frozen DSTformer representation and an affine head
fitted to model-generated pseudo-labels. The final option classifier uses question and option text together with
the newly generated multimodal prediction; identifiers and paths are excluded from its feature contract. Its
training implementation accompanies the checkpoint. The image/video backbones use the supplied pretrained
V-JEPA 2, SigLIP and SigLIP2 artifacts.

## Outputs

`submission.csv` contains the generated Large predictions. `inference_manifest.json`
records output and input identities; `runtime.json` records the numerical environment.
The run directory contains raw features, intermediate predictions, structural stage outputs,
the classifier output and their evidence. The technical report describes held-subject validation separately
from Kaggle scoring.